## Pytorch fundamentals

In [2]:
import torch

In [3]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])

In [4]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Device:", device)

Device: cuda


In [5]:
M = X.to(device)
print(M)
print(X)

tensor([[1., 4., 7.],
        [2., 3., 6.]], device='cuda:0')
tensor([[1., 4., 7.],
        [2., 3., 6.]])


In [6]:
M = torch.rand((1000, 1000)) # on the CPU
%timeit M @ M.T

7.51 ms ± 30 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
M = torch.rand((1000, 1000), device="cuda") # on the GPU
%timeit M @ M.T

347 μs ± 6.04 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Linear regression

### Load the data

In [8]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [9]:
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, random_state=42)

In [10]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [11]:
y_train.shape

(11610,)

In [12]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [13]:
y_train.shape

torch.Size([11610, 1])

### Parameters

In [14]:
torch.manual_seed(42)
n_features = X_train.shape[1] # there are 8 input features
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [15]:
learning_rate = 0.04
n_epochs = 20

for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")


Epoch 1/20, Loss: 16.158456802368164
Epoch 2/20, Loss: 12.39930248260498
Epoch 3/20, Loss: 9.638795852661133
Epoch 4/20, Loss: 7.598361015319824
Epoch 5/20, Loss: 6.079288005828857
Epoch 6/20, Loss: 4.939499378204346
Epoch 7/20, Loss: 4.0770978927612305
Epoch 8/20, Loss: 3.4187779426574707
Epoch 9/20, Loss: 2.9115939140319824
Epoch 10/20, Loss: 2.517141819000244
Epoch 11/20, Loss: 2.2074289321899414
Epoch 12/20, Loss: 1.9619368314743042
Epoch 13/20, Loss: 1.7655370235443115
Epoch 14/20, Loss: 1.6069979667663574
Epoch 15/20, Loss: 1.4779208898544312
Epoch 16/20, Loss: 1.3719767332077026
Epoch 17/20, Loss: 1.2843552827835083
Epoch 18/20, Loss: 1.2113713026046753
Epoch 19/20, Loss: 1.1501755714416504
Epoch 20/20, Loss: 1.098545789718628


In [16]:
w

tensor([[ 0.7807],
        [ 0.1166],
        [-0.0698],
        [-0.0268],
        [-0.1699],
        [-0.0654],
        [ 0.5880],
        [ 0.5573]], requires_grad=True)

In [17]:
b

tensor(1.6892, requires_grad=True)

### Make predictions

In [18]:
X_new = X_test[:5]

with torch.no_grad():
    y_pred = X_new @ w + b # b and w are trained

print("Predictions:\n", y_pred)

Predictions:
 tensor([[1.0444],
        [1.0327],
        [1.6319],
        [2.1293],
        [1.3578]])


## Linear Regression Using PyTorch’s High-Level API

In [19]:
import torch.nn as nn

In [20]:
torch.manual_seed(42)
model = nn.Linear(n_features, 1)

In [21]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [22]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [23]:
model.state_dict()

OrderedDict([('weight',
              tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]])),
             ('bias', tensor([0.3117]))])

In [24]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [25]:
for name, param in model.named_parameters():
    print(name, param.shape)

weight torch.Size([1, 8])
bias torch.Size([1])


### Making predictions

When using a module as a function, Pytorch calls the module's `forward()` method. For `nn.linear()` the `forward()` method computes `X
@ self.weight.T + self.bias` (where X is the input)

In [26]:
model(X_test[:5])

tensor([[-0.0421],
        [ 0.2654],
        [ 0.3940],
        [ 0.4194],
        [ 0.0510]], grad_fn=<AddmmBackward0>)

In [27]:
# This hook will execute during the forward pass every time
model._forward_hooks.clear()
hook = model.register_forward_hook(lambda module, input, output: print("Forward hook - input:", input))

In [28]:
model(X_test[:5])

Forward hook - input: (tensor([[-1.1578, -0.2867, -0.4955, -0.1662, -0.0295,  0.3890,  0.1937,  0.2870],
        [-0.7125,  0.1088, -0.1633,  0.2016,  0.1284, -0.1182, -0.2372,  0.0622],
        [-0.2156,  1.8491, -0.5798,  0.1853, -0.1043, -0.6769,  1.0089, -1.4271],
        [ 0.9667, -0.9196,  0.2775, -0.1706,  0.2562,  0.2056, -0.6401,  0.4320],
        [-0.0873,  0.4252,  0.0145, -0.1538, -0.3297, -0.2012,  0.4561, -1.1722]]),)


tensor([[-0.0421],
        [ 0.2654],
        [ 0.3940],
        [ 0.4194],
        [ 0.0510]], grad_fn=<AddmmBackward0>)

### Defining the model hyperparameters

In [29]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.04)
loss_fn = nn.MSELoss()

### Train the model

In [30]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    # Criterios == loss function. Its called criterion to differentiate it from the value returned by the loss function (the loss)
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [31]:
train_bgd(model, optimizer, loss_fn, X_train, y_train, n_epochs=20)

Forward hook - input: (tensor([[-0.1940, -1.0778, -0.9433,  ..., -0.5729,  0.9292, -1.4221],
        [ 0.7520, -1.8688,  0.4055,  ...,  0.2052, -0.9165,  1.0966],
        [-0.4147,  0.0297,  0.8181,  ..., -0.2998,  1.3087, -1.6970],
        ...,
        [-1.2233,  0.5043, -0.5160,  ...,  0.1345, -0.7198,  1.1466],
        [-0.9355,  1.8491, -0.1088,  ..., -0.0135,  0.5217, -0.1028],
        [ 0.8958,  0.1879,  0.2995,  ..., -0.1782,  1.1213, -1.3071]]),)
Epoch 1/20, Loss: 4.3378496170043945
Forward hook - input: (tensor([[-0.1940, -1.0778, -0.9433,  ..., -0.5729,  0.9292, -1.4221],
        [ 0.7520, -1.8688,  0.4055,  ...,  0.2052, -0.9165,  1.0966],
        [-0.4147,  0.0297,  0.8181,  ..., -0.2998,  1.3087, -1.6970],
        ...,
        [-1.2233,  0.5043, -0.5160,  ...,  0.1345, -0.7198,  1.1466],
        [-0.9355,  1.8491, -0.1088,  ..., -0.0135,  0.5217, -0.1028],
        [ 0.8958,  0.1879,  0.2995,  ..., -0.1782,  1.1213, -1.3071]]),)
Epoch 2/20, Loss: 3.753269672393799
Forward h

### Make predictions

In [32]:
X_new = X_test[:5]

with torch.no_grad():
    y_pred = model(X_new)

y_pred

Forward hook - input: (tensor([[-1.1578, -0.2867, -0.4955, -0.1662, -0.0295,  0.3890,  0.1937,  0.2870],
        [-0.7125,  0.1088, -0.1633,  0.2016,  0.1284, -0.1182, -0.2372,  0.0622],
        [-0.2156,  1.8491, -0.5798,  0.1853, -0.1043, -0.6769,  1.0089, -1.4271],
        [ 0.9667, -0.9196,  0.2775, -0.1706,  0.2562,  0.2056, -0.6401,  0.4320],
        [-0.0873,  0.4252,  0.0145, -0.1538, -0.3297, -0.2012,  0.4561, -1.1722]]),)


tensor([[0.7874],
        [1.3254],
        [2.0925],
        [2.2583],
        [1.7791]])

## Implementing a regression MLP

### Define model structure

In [33]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

### Train the model

In [34]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

In [35]:
train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs=20)

Epoch 1/20, Loss: 5.045480251312256
Epoch 2/20, Loss: 2.0523123741149902
Epoch 3/20, Loss: 1.0039883852005005
Epoch 4/20, Loss: 0.8570138216018677
Epoch 5/20, Loss: 0.7740675210952759
Epoch 6/20, Loss: 0.7225847244262695
Epoch 7/20, Loss: 0.6893726587295532
Epoch 8/20, Loss: 0.6669032573699951
Epoch 9/20, Loss: 0.650773823261261
Epoch 10/20, Loss: 0.6383934020996094
Epoch 11/20, Loss: 0.6281994581222534
Epoch 12/20, Loss: 0.6193400025367737
Epoch 13/20, Loss: 0.6113173365592957
Epoch 14/20, Loss: 0.6038705706596375
Epoch 15/20, Loss: 0.5968307852745056
Epoch 16/20, Loss: 0.5901117920875549
Epoch 17/20, Loss: 0.583646833896637
Epoch 18/20, Loss: 0.5774063467979431
Epoch 19/20, Loss: 0.5713554620742798
Epoch 20/20, Loss: 0.5654447674751282


## Implementing mini-batch gradient descent

In [36]:
from torch.utils.data import TensorDataset, DataLoader

In [44]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True) # pin_memory will allocate the data in page-locked memory which guarantees a fixed physical memory location in the CPU RAM

In [45]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

model = model.to(device)

In [46]:
learning_rate = 0.02
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [47]:
def train_mbgd(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            y_pred = model(X_batch)
            optimizer.zero_grad()

            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Average Loss: {avg_loss:.4f}")


In [48]:
train_mbgd(model, optimizer, criterion, train_loader, n_epochs=20)

Epoch 1/20, Average Loss: 0.7004
Epoch 2/20, Average Loss: 0.4478
Epoch 3/20, Average Loss: 0.4040
Epoch 4/20, Average Loss: 0.3843
Epoch 5/20, Average Loss: 0.3705
Epoch 6/20, Average Loss: 0.3647
Epoch 7/20, Average Loss: 0.3569
Epoch 8/20, Average Loss: 0.3524
Epoch 9/20, Average Loss: 0.3498
Epoch 10/20, Average Loss: 0.3476
Epoch 11/20, Average Loss: 0.3425
Epoch 12/20, Average Loss: 0.3394
Epoch 13/20, Average Loss: 0.3354
Epoch 14/20, Average Loss: 0.3312
Epoch 15/20, Average Loss: 0.3308
Epoch 16/20, Average Loss: 0.3288
Epoch 17/20, Average Loss: 0.3268
Epoch 18/20, Average Loss: 0.3249
Epoch 19/20, Average Loss: 0.3286
Epoch 20/20, Average Loss: 0.3230
